<a href="https://colab.research.google.com/github/DymaStar/DTA_2026/blob/main/ML/DS220626HW_logreg_pipeline_TASKS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Воркбук: логістична регресія + Pipeline

Просте тренування на дві теми:
- **Логістична регресія** - класика класифікації, що дає ймовірності й інтерпретовні коефіцієнти.
- **Pipeline** - складаємо препроцесинг (масштабування + кодування) і модель в один надійний конвеєр.

**Набір даних:** клієнти сервісу (`clients`). Ціль - `upgraded` (1 = перейшов на преміум, 0 = ні).

| Стовпець | Що це | Тип |
|---|---|---|
| `age` | вік | число |
| `tenure` | місяців із сервісом | число |
| `usage` | годин/міс використання | число |
| `support` | звернень у підтримку | число |
| `plan` | тариф (базовий/стандарт/сімейний) | категорія |
| `region` | регіон | категорія |
| `upgraded` | перейшов на преміум - **ціль** | 0/1 |

**Як працювати:** запусти «Підготовку даних», іди по кроках, заповнюй `# TODO`. Підказки - під кожним кроком.


---

## 🔧 Підготовка даних (просто запусти)

In [1]:
# ▶️ Просто запусти цю комірку — вона готує дані. Міняти нічого не треба.
import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 30)

# Задача: чи перейде клієнт на ПРЕМІУМ-підписку (1 = так, 0 = ні)
N = 900
age          = np.random.randint(18, 70, N)                       # вік
tenure       = np.random.randint(1, 60, N)                        # місяців із сервісом
usage        = np.random.normal(80, 35, N).clip(0, 220).round(0)  # годин/міс використання
support      = np.random.poisson(1.3, N)                          # звернень у підтримку

plan   = np.random.choice(["базовий", "стандарт", "сімейний"], N, p=[.45, .35, .20])
plan_bonus = pd.Series({"базовий": -0.4, "стандарт": 0.3, "сімейний": 1.1})

region = np.random.choice(["північ", "південь", "схід", "захід"], N)
region_bonus = pd.Series({"північ": 0.1, "південь": -0.1, "схід": 0.0, "захід": 0.2})

logit = (0.03*usage + 0.045*tenure - 0.35*support - 0.012*age
         + plan_bonus[plan].values + region_bonus[region].values
         - 3.0 + np.random.normal(0, 0.8, N))
upgraded = (logit > 0).astype(int)

clients = pd.DataFrame({
    "age": age, "tenure": tenure, "usage": usage.astype(int), "support": support,
    "plan": plan, "region": region, "upgraded": upgraded,
})

print("✅ Дані готові. Таблиця clients:", clients.shape)
print("Частка тих, хто перейшов на преміум:", f"{clients['upgraded'].mean():.0%}")

✅ Дані готові. Таблиця clients: (900, 7)
Частка тих, хто перейшов на преміум: 48%


In [2]:
# Подивись на дані
clients.head()

,age,tenure,usage,support,plan,region,upgraded
0,56,17,79,4,базовий,схід,0
1,69,5,61,2,базовий,південь,0
2,46,29,24,0,базовий,північ,0
3,32,4,100,0,стандарт,захід,1
4,60,10,52,0,стандарт,захід,0


---
### Крок 1. Розвідка: баланс класів і типи ознак
Виведи частку кожного класу в `upgraded` і визнач, які стовпці числові, а які категорійні.

*Підказка:* `clients["upgraded"].value_counts(normalize=True)`.

In [3]:
# КРОК 1. Баланс класів і типи ознак
# Логіка:
# 1) value_counts(normalize=True) показує частку класів 0 і 1.
# 2) Це важливо для класифікації: якщо класів дуже нерівно, accuracy може обманювати.
# 3) select_dtypes допомагає автоматично відділити числові та категорійні колонки.

# Частка кожного класу:
# 0 = не перейшов на преміум
# 1 = перейшов на преміум
class_balance = clients["upgraded"].value_counts(normalize=True).sort_index()

print("Баланс класів у target upgraded:")
display(class_balance)

# Визначаємо числові ознаки.
# upgraded НЕ включаємо, бо це цільова змінна, а не ознака для навчання.
num_cols = clients.drop(columns="upgraded").select_dtypes(include="number").columns.tolist()

# Визначаємо категорійні ознаки.
cat_cols = clients.drop(columns="upgraded").select_dtypes(include="object").columns.tolist()

print("Числові колонки:", num_cols)
print("Категорійні колонки:", cat_cols)

Баланс класів у target upgraded:


,proportion
upgraded,
0,0.515556
1,0.484444


Числові колонки: ['age', 'tenure', 'usage', 'support']
Категорійні колонки: ['plan', 'region']


✍️ Випиши списки стовпців (знадобляться далі):

> числові: `age`, `tenure`, `usage`, `support`  
> категорійні: `plan`, `region`

**Логіка:** числові ознаки масштабуємо, категорійні — кодуємо через One-Hot.

### Крок 2. X, y і поділ train / test
- `X` — усі стовпці, КРІМ `upgraded`. `y` — `upgraded`.
- Поділ: 20% у тест, `random_state=RANDOM_STATE`, **`stratify=y`** (щоб пропорція класів збереглась).

*Підказка:* `train_test_split(X, y, test_size=.., random_state=.., stratify=..)`.

In [4]:
from sklearn.model_selection import train_test_split

# КРОК 2. Створюємо X, y і ділимо дані на train/test
# Логіка:
# X — це таблиця ознак, тобто все, за чим модель буде робити прогноз.
# y — це цільова змінна, тобто правильна відповідь, яку модель має навчитися передбачати.

# Беремо всі колонки, крім target upgraded.
X = clients.drop(columns="upgraded")

# Target: 1 = перейшов на преміум, 0 = не перейшов.
y = clients["upgraded"]

# Ділимо дані:
# - 80% на навчання;
# - 20% на тест;
# - random_state потрібен, щоб результат повторювався;
# - stratify=y зберігає пропорцію класів 0/1 у train і test.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Баланс y_train:")
display(y_train.value_counts(normalize=True).sort_index())
print("Баланс y_test:")
display(y_test.value_counts(normalize=True).sort_index())

X_train: (720, 6)
X_test: (180, 6)
Баланс y_train:


,proportion
upgraded,
0,0.515278
1,0.484722


Баланс y_test:


,proportion
upgraded,
0,0.516667
1,0.483333


### Крок 3. Опиши, що робити з кожним типом стовпців (`ColumnTransformer`)
Числові — **масштабувати** (`StandardScaler`); категорійні — **One-Hot** (`OneHotEncoder`).
Логістичній регресії масштабування потрібне (у ній є регуляризація).

*Підказка:*
```python
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_cols = [..]
cat_cols = [..]

preprocess = ColumnTransformer([
    ("num", .., num_cols),
    ("cat", .., cat_cols),
])
```

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# КРОК 3. ColumnTransformer: різна обробка для різних типів колонок

# Логіка:
# - LogisticRegression працює з числами, тому категорії треба перетворити на числа.
# - Числові ознаки мають різні масштаби, тому застосовуємо StandardScaler.
# - Категорійні ознаки plan і region перетворюємо через OneHotEncoder.
# - handle_unknown='ignore' захищає від помилки, якщо в новому клієнті з'явиться нова категорія.

num_cols = ["age", "tenure", "usage", "support"]
cat_cols = ["plan", "region"]

preprocess = ColumnTransformer([
    # Для числових колонок: стандартизація.

    ("num", StandardScaler(), num_cols),

    # Для категорійних колонок: One-Hot Encoding.

    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

preprocess

ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['age', 'tenure', 'usage', 'support']),
                                ('cat', OneHotEncoder(handle_unknown='ignore'),
                                 ['plan', 'region'])])

### Крок 4. Збери повний `Pipeline`: препроцесинг + модель
Поклади `preprocess` і `LogisticRegression(max_iter=1000)` в один `Pipeline`.

*Підказка:*
```python
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ("prep", ..),
    ("model", ..),
])
```

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# КРОК 4. Pipeline = preprocess + модель
# Логіка:
# Pipeline потрібен, щоб усі кроки виконувалися в правильному порядку:
# 1) спочатку ColumnTransformer готує дані;
# 2) потім LogisticRegression навчається на підготовлених даних.
#
# Перевага:
# нам не треба окремо робити scaler.fit_transform(), encoder.fit_transform() тощо.
# Pipeline сам виконає fit/transform правильно.

pipe = Pipeline([
    # Крок 1: попередня обробка колонок
    ("prep", preprocess),

    # Крок 2: модель логістичної регресії
    # max_iter=1000 дає моделі більше ітерацій, щоб стабільно зійтися.
    ("model", LogisticRegression(max_iter=1000))
])

pipe

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'tenure', 'usage',
                                                   'support']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['plan', 'region'])])),
                ('model', LogisticRegression(max_iter=1000))])

### Крок 5. Навчи конвеєр і виміряй accuracy на тесті
Один виклик `.fit(X_train, y_train)` прожене дані через усі кроки.

*Підказка:* `pipe.fit(...)`, далі `pipe.score(X_test, y_test)`.

In [7]:
# КРОК 5. Навчання Pipeline і перевірка accuracy

# Логіка:
# pipe.fit(X_train, y_train):
# - навчає scaler тільки на train;
# - навчає OneHotEncoder тільки на train;
# - перетворює train;
# - навчає LogisticRegression.
#
# pipe.score(X_test, y_test):
# - автоматично застосовує вже навчені перетворення до test;
# - рахує accuracy.

pipe.fit(X_train, y_train)

accuracy = pipe.score(X_test, y_test)

print("Accuracy на тесті:", round(accuracy, 3))

Accuracy на тесті: 0.844


### Крок 6. Деталізована оцінка: матриця плутанини й звіт
Передбач класи на тесті, побудуй `confusion_matrix` і `classification_report`.

*Підказка:* `pipe.predict(X_test)`; `confusion_matrix(...)`; `classification_report(...)`.

In [8]:
from sklearn.metrics import confusion_matrix, classification_report

# КРОК 6. Деталізована оцінка моделі

# Логіка:
# Accuracy дає одну загальну цифру, але не показує, де саме модель помиляється.
# Тому додатково дивимося:
# - confusion_matrix: скільки 0/1 передбачено правильно або неправильно;
# - classification_report: precision, recall, f1-score для кожного класу.

# Прогноз класів на тестових даних.
y_pred = pipe.predict(X_test)

# Матриця плутанини:
# рядки = справжні класи, колонки = передбачені класи.
cm = confusion_matrix(y_test, y_pred)

print("Матриця плутанини:")
print(cm)

print("\nClassification report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["не перейшов", "перейшов"]
))

Матриця плутанини:
[[80 13]
 [15 72]]

Classification report:
              precision    recall  f1-score   support

 не перейшов       0.84      0.86      0.85        93
    перейшов       0.85      0.83      0.84        87

    accuracy                           0.84       180
   macro avg       0.84      0.84      0.84       180
weighted avg       0.84      0.84      0.84       180



### ✏️ Висновок

Модель показала **accuracy = 0.84**, тобто правильно класифікувала **84%** клієнтів тестової вибірки.

Матриця плутанини показує, що модель правильно визначила **80** клієнтів, які не перейшли на преміум, і **72** клієнтів, які перейшли. При цьому було допущено **28** помилок (13 хибнопозитивних і 15 хибнонегативних).

Метрики **precision**, **recall** та **F1-score** для обох класів знаходяться на рівні **0.83–0.85**, що свідчить про збалансовану якість моделі без суттєвого перекосу в бік одного з класів. Загалом модель добре підходить для прогнозування переходу клієнтів на преміум.

### Крок 7. Ймовірності + ROC-AUC
Логістична регресія дає не лише мітку, а й **ймовірність**. Дістань ймовірність класу «1» і порахуй ROC-AUC.

*Підказка:* `proba = pipe.predict_proba(X_test)[:, 1]`; `roc_auc_score(y_test, proba)`.

In [9]:
from sklearn.metrics import roc_auc_score

# КРОК 7. Ймовірності та ROC-AUC

#
# Логіка:
# predict() дає готове рішення 0 або 1.
# predict_proba() дає ймовірності класів.
# Для бізнесу ймовірність часто корисніша, бо можна змінювати поріг рішення.
#
# [:, 1] означає: беремо ймовірність саме класу 1,
# тобто ймовірність переходу на преміум.

proba = pipe.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, proba)

print("ROC-AUC:", round(roc_auc, 3))

ROC-AUC: 0.927


### Крок 8. 🔑 Інтерпретація коефіцієнтів
Дістань назви ознак після препроцесингу й коефіцієнти моделі. Знак: **+ підвищує** ймовірність переходу, − знижує.

*Підказка:*
```python
names = ..
coefs = ..
```
Зведи у `DataFrame` і відсортуй за модулем.

In [10]:
# КРОК 8. Інтерпретація коефіцієнтів Logistic Regression
#
# Логіка:
# LogisticRegression має коефіцієнти для кожної ознаки.
# Після OneHotEncoder кількість ознак змінюється:
# наприклад plan перетворюється на plan_базовий, plan_стандарт, plan_сімейний.
#
# Додатний coef:
# - ознака підвищує ймовірність класу 1, тобто переходу на преміум.
#
# Від'ємний coef:
# - ознака знижує ймовірність переходу на преміум.
#
# abs_coef потрібен, щоб відсортувати не за знаком, а за силою впливу.

# Назви ознак після StandardScaler та OneHotEncoder.
feature_names = pipe.named_steps["prep"].get_feature_names_out()

# Коефіцієнти навченої LogisticRegression.
coefs = pipe.named_steps["model"].coef_[0]

# Таблиця для інтерпретації.
coef_df = pd.DataFrame({
    "feature": feature_names,
    "coef": coefs
})

# Додаємо модуль коефіцієнта, щоб бачити силу впливу.
coef_df["abs_coef"] = coef_df["coef"].abs()

# Сортуємо: найсильніші фактори зверху.
coef_df = coef_df.sort_values("abs_coef", ascending=False)

display(coef_df)

,feature,coef,abs_coef
2,num__usage,2.027135,2.027135
1,num__tenure,1.648386,1.648386
6,cat__plan_сімейний,1.411953,1.411953
4,cat__plan_базовий,-1.339728,1.339728
3,num__support,-0.835947,0.835947
7,cat__region_захід,0.537637,0.537637
0,num__age,-0.406875,0.406875
10,cat__region_схід,-0.340620,0.340620
8,cat__region_південь,-0.189350,0.189350
9,cat__region_північ,0.136239,0.136239


✍️ **Відповідь словами:**

> Найсильніше підвищує шанс переходу на преміум `num__usage` — тобто більше використання сервісу.  
> Також шанс підвищують `num__tenure` і тариф `cat__plan_сімейний`.  
> Найсильніше знижує шанс переходу `cat__plan_базовий`, а також `num__support` — більше звернень у підтримку пов’язане з нижчою ймовірністю переходу.

**Логіка читання коефіцієнтів:**  
- `+` означає: ознака збільшує ймовірність класу `1`, тобто переходу на преміум;  
- `-` означає: ознака зменшує ймовірність класу `1`;  
- `abs_coef` показує силу впливу незалежно від знаку.

### ✏️ Інтерпретація коефіцієнтів Logistic Regression

Коефіцієнти моделі показують напрямок і силу впливу кожної ознаки на ймовірність переходу клієнта на преміум.

Найбільший позитивний вплив має **`num_usage`** (+2.03): чим активніше клієнт користується сервісом, тим вища ймовірність оформлення преміум. Також суттєво підвищують ймовірність переходу **тривалість користування (`num_tenure`)** та **сімейний тариф (`cat_plan_сімейний`)**.

Найбільший негативний вплив має **базовий тариф (`cat_plan_базовий`)** (-1.34). Крім того, часті звернення до служби підтримки (`num_support`) також знижують імовірність переходу, що може свідчити про проблеми або незадоволеність клієнтів.

Регіон проживання та вік мають помірний або слабкий вплив. Найбільш помітний серед регіонів — **західний**, який дещо підвищує ймовірність переходу, тоді як інші регіони впливають значно слабше.

Загалом модель показує, що найважливішими факторами переходу на преміум є **активність користування сервісом, тривалість користування та тип тарифного плану**.

### Крок 9. Прогноз для нового клієнта
Конвеєр приймає **сирі** дані — кодувати/масштабувати вручну не треба. Створи клієнта й виведи і рішення, і ймовірність.

Клієнт: вік 30, tenure 24, usage 120, support 0, plan «сімейний», region «захід».

*Підказка:* `pd.DataFrame([{...}])` з тими самими назвами стовпців → `pipe.predict_proba(...)[0, 1]`.

In [11]:
# КРОК 9. Прогноз для нового клієнта
# Логіка:
# Важливо: передаємо "сирі" дані в такому самому форматі, як X.
# Не треба вручну масштабувати age/tenure/usage/support.
# Не треба вручну робити One-Hot для plan/region.
# Pipeline зробить усе автоматично.

new_client = pd.DataFrame([{
    "age": 30,
    "tenure": 24,
    "usage": 120,
    "support": 0,
    "plan": "сімейний",
    "region": "захід"
}])

display(new_client)

# predict() дає клас 0 або 1.
new_pred = pipe.predict(new_client)[0]

# predict_proba()[0, 1] дає ймовірність класу 1.
new_proba = pipe.predict_proba(new_client)[0, 1]

print(
    "Рішення моделі:",
    "ПЕРЕЙДЕ на преміум" if new_pred == 1 else "НЕ перейде на преміум"
)

print("Ймовірність переходу на преміум:", round(new_proba, 3))

,age,tenure,usage,support,plan,region
0,30,24,120,0,сімейний,захід


Рішення моделі: ПЕРЕЙДЕ на преміум
Ймовірність переходу на преміум: 0.995


### Крок 10. Чесна оцінка: крос-валідація всього конвеєра
Прожени `pipe` через `cross_val_score` (cv=5, scoring="roc_auc"). Бо весь препроцесинг усередині Pipeline — кожен фолд обробляється окремо, **без витоку**.

*Підказка:* `cross_val_score(pipe, X, y, cv=5, scoring="roc_auc")`.

In [12]:
from sklearn.model_selection import cross_val_score

# КРОК 10. Крос-валідація всього Pipeline
#
# Логіка:
# Звичайний train/test split залежить від одного конкретного розбиття.
# Cross-validation перевіряє модель на кількох різних розбиттях.
#
# Тут важливо, що ми передаємо в cross_val_score саме pipe, а не окремо підготовлені дані.
# Це означає:
# - у кожному fold StandardScaler навчається тільки на train-частині цього fold;
# - OneHotEncoder теж навчається тільки на train-частині;
# - test-частина fold не "підглядається".
#
# Саме це називається "без витоку даних" / no data leakage.

cv_auc = cross_val_score(
    pipe,
    X,
    y,
    cv=5,
    scoring="roc_auc"
)

print("ROC-AUC по fold:")
print(np.round(cv_auc, 3))

print(
    f"Середній ROC-AUC: {cv_auc.mean():.3f} ± {cv_auc.std():.3f}"
)

ROC-AUC по fold:
[0.934 0.931 0.94  0.891 0.921]
Середній ROC-AUC: 0.923 ± 0.018


### ✏️ Висновок

За результатами 5-fold Cross Validation модель отримала середній показник **ROC-AUC = 0.923 ± 0.018**.

Усі п'ять розбиттів дали близькі результати (від **0.891** до **0.940**), що свідчить про стабільну роботу моделі та відсутність значних коливань її якості.

Оскільки середній ROC-AUC перевищує **0.9**, модель дуже добре розрізняє клієнтів, які перейдуть на преміум, і тих, які не перейдуть. Невелике стандартне відхилення (**±0.018**) також підтверджує, що модель добре узагальнює дані і не залежить від конкретного розбиття вибірки.

---
# ⭐ Бонус (необов'язково)
1. **Навіщо масштабування?** Збери другий конвеєр **без** `StandardScaler` (числові — `passthrough`) і порівняй ROC-AUC. Сильно змінилось?
```python
("num", "passthrough", num_cols)
```
2. **Дисбаланс класів.** Додай у `LogisticRegression(class_weight="balanced")` і подивись, як зміняться recall для класу «1» та матриця плутанини.
```python
LogisticRegression(max_iter=1000, class_weight="balanced")
```
3. **Поріг рішення.** Замість порогу 0.5 спробуй 0.3 (`proba >= 0.3`). Як зміняться precision і recall?
```python
# 3. Поріг 0.3 замість 0.5
import numpy as np
proba = pipe.predict_proba(X_test)[:, 1]
for thr in [0.5, 0.3]:
    pred_thr = (proba >= thr).astype(int)
    cm = confusion_matrix(y_test, pred_thr)
    print(f"Поріг {thr}: матриця\n{cm}")
print("→ нижчий поріг ловить більше 'так' (вищий recall), але росте й хибних тривог (нижчий precision)")
```

In [13]:
# Місце для бонусних експериментів
# Нижче можна запускати додаткові перевірки після виконання основних кроків.

# ----------------------------------------------------------
# Бонус 1. Pipeline без StandardScaler
# ----------------------------------------------------------
# Аналогія:
# у ML2-файлі комірки 3–7 пояснювали, навіщо потрібен StandardScaler.
# Тут можна перевірити, що буде, якщо числові ознаки не масштабувати.

preprocess_no_scaler = ColumnTransformer([
    ("num", "passthrough", num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

pipe_no_scaler = Pipeline([
    ("prep", preprocess_no_scaler),
    ("model", LogisticRegression(max_iter=1000))
])

cv_auc_no_scaler = cross_val_score(
    pipe_no_scaler,
    X,
    y,
    cv=5,
    scoring="roc_auc"
)

print("Без StandardScaler:")
print(f"ROC-AUC: {cv_auc_no_scaler.mean():.3f} ± {cv_auc_no_scaler.std():.3f}")


# ----------------------------------------------------------
# Бонус 2. class_weight='balanced'
# ----------------------------------------------------------
# Логіка:
# якщо клас 1 зустрічається рідше, модель може гірше його ловити.
# class_weight='balanced' дає більшій вазі рідкісному класу.

pipe_balanced = Pipeline([
    ("prep", preprocess),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

pipe_balanced.fit(X_train, y_train)

pred_balanced = pipe_balanced.predict(X_test)

print("\nBalanced model:")
print(confusion_matrix(y_test, pred_balanced))
print(classification_report(
    y_test,
    pred_balanced,
    target_names=["не перейшов", "перейшов"]
))


# ----------------------------------------------------------
# Бонус 3. Пороги рішення 0.5 і 0.3
# ----------------------------------------------------------
# predict() за замовчуванням використовує поріг 0.5.
# Але для бізнесу можна змінювати поріг:
# - нижчий поріг ловить більше потенційних клієнтів класу 1;
# - але може збільшити кількість помилкових спрацьовувань.

proba_test = pipe.predict_proba(X_test)[:, 1]

for threshold in [0.5, 0.3]:
    pred_threshold = (proba_test >= threshold).astype(int)

    print(f"\nПоріг {threshold}:")
    print(confusion_matrix(y_test, pred_threshold))
    print(classification_report(
        y_test,
        pred_threshold,
        target_names=["не перейшов", "перейшов"]
    ))

Без StandardScaler:
ROC-AUC: 0.923 ± 0.017

Balanced model:
[[80 13]
 [15 72]]
              precision    recall  f1-score   support

 не перейшов       0.84      0.86      0.85        93
    перейшов       0.85      0.83      0.84        87

    accuracy                           0.84       180
   macro avg       0.84      0.84      0.84       180
weighted avg       0.84      0.84      0.84       180


Поріг 0.5:
[[80 13]
 [15 72]]
              precision    recall  f1-score   support

 не перейшов       0.84      0.86      0.85        93
    перейшов       0.85      0.83      0.84        87

    accuracy                           0.84       180
   macro avg       0.84      0.84      0.84       180
weighted avg       0.84      0.84      0.84       180


Поріг 0.3:
[[71 22]
 [ 8 79]]
              precision    recall  f1-score   support

 не перейшов       0.90      0.76      0.83        93
    перейшов       0.78      0.91      0.84        87

    accuracy                           0.

### ✏️ Висновок

Порівняння моделей показало, що використання або відсутність `StandardScaler` практично не вплинуло на якість логістичної регресії: середній ROC-AUC залишився **0.923**, а всі основні метрики не змінилися.

При стандартному порозі **0.5** модель демонструє збалансовані результати: **Accuracy = 0.84**, **Precision ≈ 0.85** та **Recall ≈ 0.83**.

Після зниження порога до **0.3** модель почала знаходити більше клієнтів, які дійсно переходять на преміум (**Recall зріс з 0.83 до 0.91**), однак збільшилася кількість хибнопозитивних прогнозів, через що **Precision знизився з 0.85 до 0.78**.

Отже, вибір порога залежить від бізнес-цілі: якщо важливо знайти якомога більше потенційних преміум-клієнтів, доцільно використовувати нижчий поріг (**0.3**). Якщо ж важливіше мінімізувати кількість помилкових пропозицій, оптимальним залишається стандартний поріг **0.5**.

---
# 🧠 Питання на розуміння (без коду)
1. Чому логістична регресія — це **класифікація**, попри слово «регресія» в назві?
2. Що показує `predict_proba` і чим воно корисніше за `predict` для бізнесу?
3. Навіщо взагалі загортати кроки в `Pipeline` — що поганого станеться, якщо масштабувати дані **до** `train_test_split`?
4. Логістичній регресії масштабування потрібне, а дереву рішень — ні. Чому?
5. Коефіцієнт `support` від'ємний. Як прочитати це вголос для керівника?

> 🎯 Якщо зібрав робочий Pipeline і впевнено читаєш коефіцієнти — ти володієш найбільш «продакшн-готовим» патерном класичного ML.

# 🧠 Питання на розуміння — відповіді

### 1. Чому логістична регресія — це класифікація, попри слово «регресія» в назві?

Назва історична: модель будує лінійну комбінацію ознак (як у звичайній регресії), але потім пропускає її через сигмоїду, щоб отримати ймовірність від 0 до 1, і вже за порогом (зазвичай 0.5) перетворює це на клас 0/1. Тобто регресується не сам клас, а логарифм шансів (log-odds), а результат — класифікаційний.

### 2. Що показує `predict_proba` і чим воно корисніше за `predict` для бізнесу?

`predict_proba` повертає ймовірність належності до кожного класу, а не лише фінальну мітку. Бізнесу це дає гнучкість:
- можна сортувати клієнтів за ймовірністю (наприклад, для таргетованої розсилки топ-20% найімовірніших до апгрейду);
- можна змінювати поріг рішення під вартість помилок (як у бонусі з порогом 0.3);
- можна оцінювати впевненість моделі, а не просто отримувати бінарний вердикт.

### 3. Навіщо взагалі загортати кроки в `Pipeline` — що поганого станеться, якщо масштабувати дані до `train_test_split`?

Якщо зробити `StandardScaler().fit()` на всіх даних до спліту, то середнє і стандартне відхилення порахуються з урахуванням тестових рядків — тобто модель «підглядає» статистику тесту ще до навчання. Це називається **data leakage** (витік даних): метрики на тесті виявляться завищеними й нереалістичними, бо тест більше не є по-справжньому «незнайомими» даними.

`Pipeline` гарантує, що `fit` препроцесингу відбувається тільки на train у кожному розбитті — це й видно у кроці 10, де кожен fold cross-validation обробляється окремо, без витоку.

### 4. Логістичній регресії масштабування потрібне, а дереву рішень — ні. Чому?

Логрегресія використовує регуляризацію (штраф на величину коефіцієнтів) і градієнтну оптимізацію — обидва процеси чутливі до масштабу ознак: якщо одна ознака (наприклад, `usage` зі значеннями до 220) набагато більша за іншу (`support` зі значеннями 0–5), регуляризація несправедливо «карає» їх по-різному, а оптимізатор збігається повільніше.

Дерево рішень натомість шукає поріг розбиття для кожної ознаки окремо (наприклад, `usage > 75`) — масштаб значення не впливає на те, де саме проходить розбиття, тож нормалізація йому не потрібна.

### 5. Коефіцієнт `support` від'ємний. Як прочитати це вголос для керівника?

«Що більше клієнт звертається у підтримку, то нижча ймовірність, що він перейде на преміум — кожне додаткове звернення знижує шанс апгрейду. Ймовірно, часті звернення сигналізують про незадоволеність сервісом, тож варто звернути увагу на якість підтримки для таких клієнтів, перш ніж пропонувати їм преміум.»